# 04 â€” Apply HMRF Spatial Regularization

Refine GMM labels using a Hidden Markov Random Field with Potts prior. Neighboring voxels are encouraged to share the same label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gc

from research_ct.io.volume_saver import Load_From_Numpy, Load_From_Numpy_Slab
from research_ct.segmentation.hmrf import Hmrf_Segmenter


In [ ]:
# Load GMM outputs — both .npz and .npy are lazy (memmap).
# No data enters RAM until explicitly sliced/indexed.
Processed = Load_From_Numpy(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\processed\preprocessed_volume.npz"
)
Probs_Gmm = Load_From_Numpy(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy"
)

D, H, W, K = Probs_Gmm.shape
print(f"Processed : {Processed.shape}")
print(f"Probs_Gmm : {Probs_Gmm.shape}  K={K}")

# Derive GMM labels from probabilities (argmax along last axis).
# This reads the full memmap once — (D,H,W) int32 ~ {D*H*W*4/1e9:.1f} GB.
Labels_Gmm = Probs_Gmm.argmax(axis=-1).astype(np.int32)
print(f"Labels_Gmm : {Labels_Gmm.shape}  ({Labels_Gmm.nbytes / 1e9:.2f} GB)")


## Compute Log-Probabilities

HMRF needs log p(x_i | k) for the energy function.

In [ ]:
# Load only the test Z-range as a concrete float32 array.
# Load_From_Numpy_Slab reads exactly the requested slices from disk —
# the rest of the 24 GB probabilities file stays on disk.
Test_Z_Range = slice(0, min(50, D))
Num_Test_Slices = Test_Z_Range.stop - (Test_Z_Range.start or 0)

Probs_Slice = Load_From_Numpy_Slab(
    r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy",
    Test_Z_Range.start, Test_Z_Range.stop,
    dtype=np.float32,
)

# Compute log in-place to avoid a second (Num_Test_Slices, H, W, K) allocation
np.clip(Probs_Slice, 1e-10, 1.0, out=Probs_Slice)
np.log(Probs_Slice, out=Probs_Slice)  # Probs_Slice is now Log_Probs — no copy made
Log_Probs_Slice = Probs_Slice  # rename for clarity, same object

print(f"Log-probs range : [{Log_Probs_Slice.min():.2f}, {Log_Probs_Slice.max():.2f}]")


## Run HMRF

ICM optimization with Potts prior. Beta controls spatial smoothness strength.

- Low beta (~0.1): weak smoothing, preserves fine details
- High beta (~2.0): strong smoothing, removes noise

Start with beta=0.5 and adjust based on results.

In [ ]:
Hmrf = Hmrf_Segmenter(
    Beta=0.5,
    Max_Iterations=20,
    Connectivity=6,
)

# Volume argument removed â€” shape comes from Log_Probabilities directly
Labels_Hmrf = Hmrf.Fit(Log_Probs_Slice)

del Log_Probs_Slice, Probs_Slice
gc.collect()

## Compare GMM vs HMRF (Test Region)

Visualize the effect of spatial regularization.

In [ ]:
# Derive GMM labels on the fly — each slice loaded from disk individually.
# Load_From_Numpy_Slab reads exactly one Z-slice per iteration.
fig, axes = plt.subplots(3, 3, figsize=(15, 15), constrained_layout=True)
probs_path = r"C:\Users\gabri\Documento\Mitacs\research_ct\data\output\gmm_probabilities.npy"

for i, z in enumerate([10, 25, 40]):
    Prob_Slice_Z = Load_From_Numpy_Slab(probs_path, z, z + 1, dtype=np.float32)
    Prob_Slice_Z = Prob_Slice_Z[0]  # drop the Z=1 singleton dim → (H, W, K)
    Labels_Gmm_Z = Prob_Slice_Z.argmax(axis=-1)  # (H, W) derived, no file

    axes[0, i].imshow(Processed[z], cmap="gray")
    axes[0, i].set_title(f"Processed — Z={z}")
    axes[0, i].axis("off")

    axes[1, i].imshow(Labels_Gmm_Z, cmap="tab10", vmin=0, vmax=K - 1)
    axes[1, i].set_title(f"GMM — Z={z}")
    axes[1, i].axis("off")

    axes[2, i].imshow(Labels_Hmrf[z], cmap="tab10", vmin=0, vmax=K - 1)
    axes[2, i].set_title(f"HMRF — Z={z}")
    axes[2, i].axis("off")

    del Prob_Slice_Z, Labels_Gmm_Z

axes[0, 0].set_ylabel("Intensity", fontsize=12)
axes[1, 0].set_ylabel("GMM Only", fontsize=12)
axes[2, 0].set_ylabel("HMRF", fontsize=12)
plt.suptitle("Spatial Regularization Comparison", fontsize=14)
fig.savefig("../data/output/hmrf_comparison.png", dpi=150)
plt.close(fig)
print("Saved → hmrf_comparison.png")


## Run on Full Volume (if satisfied with beta)

Once beta is tuned, apply to the entire volume. This may take 30+ minutes.

In [ ]:
# Uncomment to run on full volume (requires loading log-probs for all Z)
# Log_Probs_Full = np.log(np.clip(np.array(Probs_Gmm, dtype=np.float32),
#                                  1e-10, 1.0, out=None))
# Hmrf_Full = Hmrf_Segmenter(Beta=0.5, Max_Iterations=50, Connectivity=6)
# Labels_Hmrf_Full = Hmrf_Full.Fit(Log_Probs_Full)
#
# Save_As_Numpy(Labels_Hmrf_Full.astype(np.uint8),
#                "../data/output/hmrf_labels.npz")

print("Uncomment the cell above to run HMRF on full volume")


## Quantitative Comparison

Compare label statistics before and after HMRF.

In [ ]:
from research_ct.analysis.material_stats import Compute_Material_Statistics

# GMM stats
Stats_Gmm = Compute_Material_Statistics(Processed, Labels_Gmm, Num_Classes=int(Labels_Gmm.max()+1))

print("GMM Segmentation:")
print(f"  Classes found: {len(Stats_Gmm['classes'])}")
for c in Stats_Gmm['classes']:
    print(f"  Class {c['class_id']}: {c['voxel_count']:,} voxels ({c['volume_fraction']:.2%})")

# If HMRF full run completed, compare
# Stats_Hmrf = Compute_Material_Statistics(Processed, Labels_Hmrf_Full, ...)
# print("\nHMRF Segmentation:")
# ...